# Этап 3: Prompt-based классификация (Zero / One / Few-shot)

Классификация писем через промпты с генеративными LLM.

**Пайплайн:**
1. Генерация описаний 36 классов (Qwen-32B)
2. Подготовка few-shot примеров (K=1,3,5) с учётом групп A/B/C
3. Запуск экспериментов: 5 моделей x 4 режима (zero/one/few-shot)
4. Оценка и сохранение результатов
5. Визуализации

**Тест:** `data_test.csv` (343 примера) — единый для всех методов.

___
## Подготовка окружения (Запуск в Colab)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys
from pathlib import Path

# Корень проекта на Google Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/VKR/code")
sys.path.insert(0, str(PROJECT_ROOT))

## Подготовка окружения (Локальный запуск)

In [ ]:
# import sys
# from pathlib import Path
# PROJECT_ROOT = Path(".").resolve().parent
# sys.path.insert(0, str(PROJECT_ROOT))

## Импорты и конфигурация

In [ ]:
import json
import gc
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from src.utils.data_loader import load_dataset, load_test_set, TEXT_COL, LABEL_COL, RANDOM_SEED
from src.classification.few_shot_examples import (
    classify_groups, prepare_few_shot_examples, load_few_shot_examples,
)
from src.classification.prompt_classifier import (
    load_prompt_config, load_model, unload_model,
    build_prompt, classify_dataset, load_class_descriptions,
)
from src.classification.evaluate import evaluate_prompt_classification

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

DATA_DIR = PROJECT_ROOT / "Data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

In [ ]:
# Загрузка конфига prompt_classification
cfg = load_prompt_config()
gen_params = cfg["generation_params"]
prompt_models = cfg["prompt_models"]

print("Модели:")
for key, m in prompt_models.items():
    print(f"  {key}: {m['model_name']} (ctx={m['max_context']}, vram~{m['vram_gb']}GB)")

In [ ]:
# Загрузка данных
df_train_orig = load_dataset(stage=0)
df_test = load_test_set()

labels = sorted(df_train_orig[LABEL_COL].unique().tolist())
groups = classify_groups(df_train_orig)

print(f"\nТест: {len(df_test)} примеров, Классов: {len(labels)}")
print(f"Группы: A={sum(1 for g in groups.values() if g=='A')}, "
      f"B={sum(1 for g in groups.values() if g=='B')}, "
      f"C={sum(1 for g in groups.values() if g=='C')}")

___
## Шаг 1: Генерация описаний классов

Используем Qwen-32B-AWQ для генерации описаний всех 36 классов.
Описания нужны для промптов — помогают модели понять специфику каждого подразделения.

In [ ]:
desc_path = DATA_DIR / "class_descriptions.json"

if desc_path.exists():
    descriptions = load_class_descriptions(desc_path)
    print(f"Описания загружены из кэша ({len(descriptions)} классов)")
else:
    from src.augmentation.stage1_llm_generate import generate_class_context
    from src.augmentation.llm_utils import load_llm

    # Загружаем Qwen-32B для генерации описаний
    config_path = str(PROJECT_ROOT / "config_models" / "aug_configs" / "model_vllm_32b.json")
    llm, sampling_params, system_prompt = load_llm(config_path)

    descriptions = {}
    for cls in sorted(labels):
        examples = df_train_orig[df_train_orig[LABEL_COL] == cls][TEXT_COL].tolist()[:5]
        desc = generate_class_context(cls, examples, llm, sampling_params, system_prompt)
        descriptions[cls] = desc

    # Сохраняем
    with open(desc_path, "w", encoding="utf-8") as f:
        json.dump(descriptions, f, ensure_ascii=False, indent=2)

    print(f"Сгенерировано и сохранено {len(descriptions)} описаний")

    # Выгружаем Qwen-32B — дальше он не нужен
    del llm, sampling_params, system_prompt
    gc.collect()
    torch.cuda.empty_cache()

# Пример
first_cls = sorted(descriptions.keys())[0]
print(f"\n{first_cls}:\n{descriptions[first_cls]}")

___
## Шаг 2: Подготовка few-shot примеров

Для K=1, 3, 5 с учётом групп:
- **A** (9 кл., 50+) и **B** (15 кл., 15-49): примеры только из оригинального train
- **C** (12 кл., <15): из оригинального + аугментированного

In [ ]:
fs_path = DATA_DIR / "few_shot_examples.json"

if fs_path.exists():
    fs_data = load_few_shot_examples(fs_path)
    groups = fs_data["groups"]
    print("Few-shot примеры загружены из кэша")
else:
    fs_data = prepare_few_shot_examples(k_values=[1, 3, 5])
    groups = fs_data["groups"]
    print("Few-shot примеры сгенерированы и сохранены")

for k in ["1", "3", "5"]:
    total = sum(len(v) for v in fs_data["examples"][k].values())
    print(f"K={k}: {total} примеров")

___
## Шаг 3: Запуск экспериментов

Матрица: 5 моделей x 4 режима (K=0,1,3,5).

| Модель | K=0 | K=1 | K=3 | K=5 |
|---|---|---|---|---|
| Saiga-8B | + | ? | skip | skip |
| Gemma-9B | + | ? | skip | skip |
| Vikhr-12B | + | + | + | ? |
| Qwen-14B | + | + | + | ? |
| Qwen-32B-AWQ | + | + | + | ? |

In [ ]:
# Матрица экспериментов: какие K запускать для каждой модели
experiment_matrix = {
    "saiga_8b":  [0, 1],        # 8K контекст — K=3,5 не влезет
    "gemma_9b":  [0, 1],        # 8K контекст
    "vikhr_12b": [0, 1, 3, 5],  # 32K контекст
    "qwen_14b":  [0, 1, 3, 5],  # 32K+ контекст
    "qwen_32b":  [0, 1, 3, 5],  # 32K+ контекст
}

# Подхватываем уже посчитанные результаты (если ноутбук перезапущен)
results_path = RESULTS_DIR / "prompt_results.csv"
if results_path.exists():
    df_existing = pd.read_csv(results_path)
    all_results = df_existing.to_dict("records")
    done = {(r["model"], int(r["k_shots"])) for r in all_results}
    print(f"Загружено {len(all_results)} готовых экспериментов: {done}")
else:
    all_results = []
    done = set()
    print("Предыдущих результатов нет, запуск с нуля")

In [ ]:
def run_experiment(model_key, k_shot, model, tokenizer, model_cfg):
    """Запускает один эксперимент: модель x K."""
    max_context = model_cfg["max_context"]

    # Определяем режим и примеры
    if k_shot == 0:
        mode = "zero_shot"
        examples = None
    elif k_shot == 1:
        mode = "one_shot"
        examples = fs_data["examples"]["1"]
    else:
        mode = "few_shot"
        examples = fs_data["examples"][str(k_shot)]

    print(f"\n{'='*60}")
    print(f"Эксперимент: {model_key} | K={k_shot} ({mode})")
    print(f"{'='*60}")

    # Проверка длины промпта на первом примере
    test_prompt = build_prompt(
        df_test.iloc[0][TEXT_COL], labels, descriptions, mode, examples,
    )
    test_tokens = len(tokenizer.encode(test_prompt))
    print(f"Длина тестового промпта: {test_tokens} токенов (лимит: {max_context})")

    if test_tokens >= max_context - 100:
        print(f"SKIP: промпт ({test_tokens}) >= лимит ({max_context - 100})")
        return {
            "model": model_key,
            "model_name": model_cfg["model_name"],
            "model_size": model_cfg["vram_gb"],
            "k_shots": k_shot,
            "skipped": True,
            "prompt_tokens": test_tokens,
            "n_test": len(df_test),
        }

    # Классификация
    df_preds = classify_dataset(
        df_test, model, tokenizer, labels, descriptions,
        mode=mode, gen_params=gen_params,
        max_context=max_context, examples=examples,
        fuzzy_cutoff=cfg["extract_prediction"]["fuzzy_cutoff"],
    )

    # Оценка
    metrics = evaluate_prompt_classification(df_preds, groups)

    # Сохраняем предсказания для анализа
    preds_path = RESULTS_DIR / f"preds_{model_key}_k{k_shot}.csv"
    df_preds.to_csv(preds_path, index=False)

    return {
        "model": model_key,
        "model_name": model_cfg["model_name"],
        "model_size": model_cfg["vram_gb"],
        "k_shots": k_shot,
        "balanced_accuracy": metrics["balanced_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "unknown_rate": metrics["unknown_rate"],
        "f1_group_A": metrics.get("f1_group_A", None),
        "f1_group_B": metrics.get("f1_group_B", None),
        "f1_group_C": metrics.get("f1_group_C", None),
        "prompt_tokens": test_tokens,
        "skipped": False,
        "n_test": len(df_test),
    }

In [ ]:
# Основной цикл: загружаем модель, прогоняем все K, выгружаем
for model_key, k_values in experiment_matrix.items():
    model_cfg = prompt_models[model_key]

    # Фильтруем только те K, которые ещё не посчитаны
    k_todo = [k for k in k_values if (model_key, k) not in done]

    if not k_todo:
        print(f"\n[SKIP] {model_cfg['model_name']} — все K уже посчитаны")
        continue

    print(f"\n{'#'*60}")
    print(f"# МОДЕЛЬ: {model_cfg['model_name']}")
    print(f"# K для расчёта: {k_todo} (пропущено: {len(k_values) - len(k_todo)})")
    print(f"{'#'*60}")

    model, tokenizer = load_model(model_cfg)

    for k in k_todo:
        result = run_experiment(model_key, k, model, tokenizer, model_cfg)
        all_results.append(result)
        done.add((model_key, k))

        # Промежуточное сохранение после каждого эксперимента
        pd.DataFrame(all_results).to_csv(
            RESULTS_DIR / "prompt_results.csv", index=False,
        )

    unload_model(model, tokenizer)

print(f"\nВсего экспериментов: {len(all_results)}")

___
## Шаг 4: Сводная таблица результатов

In [ ]:
# Загрузка результатов (если ноутбук перезапущен)
df_prompt = pd.read_csv(RESULTS_DIR / "prompt_results.csv")
df_prompt = df_prompt[~df_prompt["skipped"].astype(bool)]

print("Результаты prompt-based классификации:")
display_cols = ["model", "k_shots", "balanced_accuracy", "macro_f1",
                "unknown_rate", "f1_group_A", "f1_group_B", "f1_group_C"]
df_prompt[display_cols].round(4)

In [ ]:
# Сводная таблица: baseline + prompt
df_baseline = pd.read_csv(RESULTS_DIR / "classification_results.csv")

# Приводим baseline к общему формату
rows = []
for _, r in df_baseline.iterrows():
    rows.append({
        "method": "baseline" if r["stage"] == "baseline" else "augmented",
        "model": r["model"],
        "setting": r["stage"],
        "balanced_accuracy": r["balanced_accuracy"],
        "macro_f1": r["macro_f1"],
        "unknown_rate": 0.0,
    })

# Добавляем prompt результаты
for _, r in df_prompt.iterrows():
    rows.append({
        "method": "prompt",
        "model": r["model_name"],
        "setting": f"K={int(r['k_shots'])}",
        "balanced_accuracy": r["balanced_accuracy"],
        "macro_f1": r["macro_f1"],
        "unknown_rate": r["unknown_rate"],
    })

df_all = pd.DataFrame(rows)
df_all.to_csv(RESULTS_DIR / "all_methods_comparison.csv", index=False)
print("Сводная таблица сохранена в results/all_methods_comparison.csv")
df_all.round(4)

___
## Шаг 5: Визуализации

In [ ]:
# 1. Барплот всех методов по macro_f1
fig, ax = plt.subplots(figsize=(16, 8))

df_plot = df_all.copy()
df_plot["label"] = df_plot["model"] + " (" + df_plot["setting"] + ")"
df_plot = df_plot.sort_values("macro_f1", ascending=True)

colors = df_plot["method"].map({"baseline": "#5B9BD5", "augmented": "#70AD47", "prompt": "#ED7D31"})
ax.barh(df_plot["label"], df_plot["macro_f1"], color=colors)
ax.set_xlabel("Macro F1")
ax.set_title("Сравнение всех методов классификации")

# Легенда
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#5B9BD5", label="Baseline"),
    Patch(facecolor="#70AD47", label="Augmented"),
    Patch(facecolor="#ED7D31", label="Prompt-based"),
]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "all_methods_barplot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 2. Кривая F1 от K для каждой модели
fig, ax = plt.subplots(figsize=(10, 6))

for model_key in experiment_matrix:
    df_m = df_prompt[df_prompt["model"] == model_key].sort_values("k_shots")
    if len(df_m) > 0:
        ax.plot(df_m["k_shots"], df_m["macro_f1"], marker="o", label=model_key, linewidth=2)

ax.set_xlabel("K (количество примеров на класс)")
ax.set_ylabel("Macro F1")
ax.set_title("Зависимость F1 от количества few-shot примеров")
ax.set_xticks([0, 1, 3, 5])
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "f1_vs_k.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 3. Метрики по группам A/B/C
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for idx, group in enumerate(["A", "B", "C"]):
    col = f"f1_group_{group}"
    df_g = df_prompt[["model", "k_shots", col]].dropna()

    for model_key in experiment_matrix:
        df_m = df_g[df_g["model"] == model_key].sort_values("k_shots")
        if len(df_m) > 0:
            axes[idx].plot(df_m["k_shots"], df_m[col], marker="o", label=model_key)

    axes[idx].set_title(f"Группа {group}")
    axes[idx].set_xlabel("K")
    axes[idx].set_xticks([0, 1, 3, 5])
    if idx == 0:
        axes[idx].set_ylabel("Macro F1")
    axes[idx].legend(fontsize=8)

plt.suptitle("F1 по группам классов (A/B/C)", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "f1_by_groups.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 4. Confusion matrix лучшего метода
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Находим лучший эксперимент
best = df_prompt.loc[df_prompt["macro_f1"].idxmax()]
best_file = RESULTS_DIR / f"preds_{best['model']}_k{int(best['k_shots'])}.csv"
print(f"Лучший: {best['model']} K={int(best['k_shots'])} (macro_f1={best['macro_f1']:.4f})")

df_best = pd.read_csv(best_file)
df_best = df_best[~df_best["skipped"].astype(bool)]

cm = confusion_matrix(df_best["true_label"], df_best["predicted_label"], labels=labels)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, xticklabels=labels, yticklabels=labels,
            fmt="d", cmap="Blues", ax=ax, annot=True, annot_kws={"size": 7})
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix: {best['model']} K={int(best['k_shots'])}")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix_best.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5. Per-class F1 с выделением групп цветом
from sklearn.metrics import f1_score as f1_per_class
from sklearn.preprocessing import LabelEncoder

df_best = pd.read_csv(best_file)
df_best = df_best[~df_best["skipped"].astype(bool)]

# Per-class F1
from sklearn.metrics import classification_report
report = classification_report(
    df_best["true_label"], df_best["predicted_label"],
    labels=labels, output_dict=True, zero_division=0,
)

per_class = pd.DataFrame({
    "class": labels,
    "f1": [report[l]["f1-score"] for l in labels],
    "group": [groups.get(l, "?") for l in labels],
}).sort_values("f1", ascending=True)

group_colors = {"A": "#5B9BD5", "B": "#70AD47", "C": "#ED7D31"}

fig, ax = plt.subplots(figsize=(14, 12))
bars = ax.barh(per_class["class"], per_class["f1"],
               color=[group_colors[g] for g in per_class["group"]])
ax.set_xlabel("F1-score")
ax.set_title(f"Per-class F1: {best['model']} K={int(best['k_shots'])}")

legend_elements = [Patch(facecolor=c, label=f"Группа {g}") for g, c in group_colors.items()]
ax.legend(handles=legend_elements, loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 6. Unknown rate по моделям
fig, ax = plt.subplots(figsize=(10, 6))

for model_key in experiment_matrix:
    df_m = df_prompt[df_prompt["model"] == model_key].sort_values("k_shots")
    if len(df_m) > 0:
        ax.plot(df_m["k_shots"], df_m["unknown_rate"], marker="s", label=model_key, linewidth=2)

ax.set_xlabel("K (количество примеров на класс)")
ax.set_ylabel("Unknown rate")
ax.set_title("Доля нераспознанных ответов по моделям")
ax.set_xticks([0, 1, 3, 5])
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "unknown_rate.png", dpi=150, bbox_inches="tight")
plt.show()

___
## Итого

Результаты сохранены:
- `results/prompt_results.csv` — все эксперименты prompt-based
- `results/all_methods_comparison.csv` — сводка всех методов
- `results/preds_<model>_k<K>.csv` — предсказания каждого эксперимента
- Визуализации в `results/`